# Qiskit basics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Broum/UKP_IQC_materials/blob/main/Qiskit_intro_colab.ipynb)

A minimal tour: install &rarr; build a circuit &rarr; draw it &rarr; Bloch sphere &rarr; run on a simulator.
Run the cells in order (`Shift+Enter`).

## 1. Installation

In Colab everything is installed per session - the cell below has to be re-run every time
the runtime restarts. `qiskit` is the SDK, `qiskit-aer` the local simulator, `pylatexenc`
is needed for nice circuit drawings.

In [ ]:
%pip install -q qiskit qiskit-aer pylatexenc matplotlib

In [ ]:
import qiskit

print(qiskit.__version__)

### Running locally instead: a conda environment

Colab is enough for this notebook, but for the course projects you will want a local
environment. [Miniconda](https://www.anaconda.com/docs/getting-started/miniconda/main) is a
minimal Anaconda: the `conda` package manager without the several GB of preinstalled
packages. Download the installer for your OS from that page and run it (on Linux/macOS:
`bash Miniconda3-latest-<your-platform>.sh`, then reopen the terminal).

Then, in a terminal - **not** in this notebook:

```bash
# create an environment with its own Python
conda create -n qiskit python=3.12

# activate it - repeat this in every new terminal
conda activate qiskit

# qiskit is not on the default conda channels, install it with pip inside the environment
pip install qiskit qiskit-aer pylatexenc matplotlib

# ipykernel lets any notebook editor run code in this environment
pip install notebook ipykernel
```

Useful afterwards:

```bash
conda env list               # list your environments
conda list                   # what is installed in the active one
conda deactivate             # leave the environment
conda env remove -n qiskit   # delete it and start over
```

Keep one environment per course/project: if you break it, you delete it and recreate it
without touching your system Python.

#### Opening a notebook in Jupyter

With the environment active, go to the folder with your `.ipynb` files and start Jupyter:

```bash
conda activate qiskit
cd path/to/your/notebooks
jupyter notebook
```

A browser tab opens with a file listing - click a notebook to open it. Because Jupyter was
started from inside the `qiskit` environment, the notebook already uses that Python and no
kernel has to be selected. Stop the server with `Ctrl+C` in the terminal when you are done.

#### Using the environment in VS Code

VS Code runs notebooks itself, so you do not start Jupyter by hand - but the environment
needs `ipykernel` (installed above) for VS Code to be able to run code in it.

1. Install the **Python** and **Jupyter** extensions from the VS Code marketplace.
2. Open the folder with your notebooks and open an `.ipynb` file.
3. Click **Select Kernel** in the top right &rarr; *Python Environments* &rarr; pick `qiskit`.

If `qiskit` is not in the list, reload VS Code (`Ctrl+Shift+P` &rarr; *Developer: Reload
Window*); a freshly created environment is sometimes not picked up until then. You can also
register it explicitly, which makes it appear in both VS Code and `jupyter notebook`:

```bash
conda activate qiskit
python -m ipykernel install --user --name qiskit --display-name "Python (qiskit)"
```

## 2. Your first quantum circuit

A Bell state: a Hadamard on qubit 0 puts it in superposition, the CNOT entangles it with qubit 1.
The result is $\frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$.

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2, 2)   # 2 qubits, 2 classical bits
qc.h(0)                     # Hadamard on qubit 0
qc.cx(0, 1)                 # CNOT: control 0, target 1
qc.measure([0, 1], [0, 1])  # measure both qubits
qc

## 3. Visualising the circuit

`draw()` accepts several backends: `'text'` (ASCII), `'mpl'` (matplotlib) and `'latex'`.

In [ ]:
print(qc.draw('text'))
qc.draw('mpl')

## 4. The Bloch sphere

A single qubit state can be drawn as a point on the Bloch sphere. We build a *measurement-free*
circuit, get its `Statevector` and plot it. Try replacing `h(0)` by `x(0)`, `ry(0.7, 0)`, ...

In [ ]:
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_bloch_multivector

one_qubit = QuantumCircuit(1)
one_qubit.h(0)  # |0> -> |+>

state = Statevector(one_qubit)
print(state)
plot_bloch_multivector(state)

In [ ]:
# The same works for many qubits - but entangled qubits have no single-qubit
# Bloch vector, so both arrows collapse to the centre of the sphere.
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
plot_bloch_multivector(Statevector(bell))

## 5. Running on a simulator

`AerSimulator` runs the circuit locally. `transpile` rewrites the circuit into the gates the
backend understands - the same step is needed before running on real hardware.

In [ ]:
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

simulator = AerSimulator()
compiled = transpile(qc, simulator)
result = simulator.run(compiled, shots=1024).result()

counts = result.get_counts()
print(counts)
plot_histogram(counts)

Roughly half `00` and half `11`, and (almost) no `01` or `10` - the two qubits are correlated.
The exact split changes every run: measurement is random, and 1024 shots is a finite sample.

**Next steps:** add gates to `qc`, change the number of shots, or try `qc.z(0)` before the
measurement and see what happens to the histogram and to the Bloch sphere.